# Load Dataset

In [3]:
from beir.datasets.data_loader import GenericDataLoader

data_path = "scifact"

corpus, queries, qrels = GenericDataLoader(
    data_path
).load(split="test")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 99684.87it/s]


In [5]:
import pandas as pd

corpus_df = pd.DataFrame.from_dict(
    corpus,
    orient="index"   
).reset_index()

In [6]:
corpus_df

,index,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [7]:
documents = [
    row['title'] + ". " + row['text'] 
    for _, row in corpus_df.iterrows()
]

# Embedding Model

In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2416.60it/s]


In [ ]:
import time

start_time = time.time()

corpus_embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

end_time = time.time()

print(f"Encoding took {end_time - start_time:.2f} seconds")
print(document_embeddings.shape)

In [11]:
corpus_df['embedding'] = list(corpus_embeddings)

# Faiss

In [12]:
import faiss

In [13]:
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(corpus_embeddings.astype('float32'))

In [14]:
index.ntotal

5183

In [15]:
queries

{'1': '0-dimensional biomaterials show inductive properties.',
 '3': '1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.',
 '5': '1/2000 in UK have abnormal PrP positivity.',
 '13': '5% of perinatal mortality is due to low birth weight.',
 '36': 'A deficiency of vitamin B12 increases blood levels of homocysteine.',
 '42': 'A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait subjects.',
 '48': 'A total of 1,000 people in the UK are asymptomatic carriers of vCJD infection.',
 '49': 'ADAR1 binds to Dicer to cleave pre-miRNA.',
 '50': 'AIRE is expressed in some skin tumors.',
 '51': 'ALDH1 expression is associated with better breast cancer outcomes.',
 '53': 'ALDH1 expression is associated with poorer prognosis in breast cancer.',
 '54': 'AMP-activated protein kinase (AMPK) activation increases inflammation-related fibrosis in the lung

In [16]:
qrels

{'1': {'31715818': 1},
 '3': {'14717500': 1},
 '5': {'13734012': 1},
 '13': {'1606628': 1},
 '36': {'5152028': 1, '11705328': 1},
 '42': {'18174210': 1},
 '48': {'13734012': 1},
 '49': {'5953485': 1},
 '50': {'12580014': 1},
 '51': {'45638119': 1},
 '53': {'45638119': 1},
 '54': {'49556906': 1},
 '56': {'4709641': 1},
 '57': {'4709641': 1},
 '70': {'5956380': 1, '4414547': 1},
 '72': {'6076903': 1},
 '75': {'4387784': 1},
 '94': {'1215116': 1},
 '99': {'18810195': 1},
 '100': {'4381486': 1},
 '113': {'6157837': 1},
 '115': {'33872649': 1},
 '118': {'6372244': 1},
 '124': {'4883040': 1},
 '127': {'21598000': 1},
 '128': {'8290953': 1},
 '129': {'27768226': 1},
 '130': {'27768226': 1},
 '132': {'7975937': 1},
 '133': {'38485364': 1,
  '6969753': 1,
  '17934082': 1,
  '16280642': 1,
  '12640810': 1},
 '137': {'26016929': 1},
 '141': {'6955746': 1, '14437255': 1},
 '142': {'10582939': 1},
 '143': {'10582939': 1},
 '146': {'10582939': 1},
 '148': {'1084345': 1},
 '163': {'18872233': 1},
 '1

In [17]:
queries_df = (pd.DataFrame.from_dict(queries, orient="index")
              .reset_index()
              .rename(columns={'index': 'query_id', 0: 'query_text'}))

queries_df

,query_id,query_text
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [18]:
qrels_list = []
for query_id, docs in qrels.items():
    for doc_id, relevance in docs.items():
        qrels_list.append({
            'query_id': query_id,
            'doc_id': doc_id,
            'relevance': relevance
        })

qrels_df = pd.DataFrame(qrels_list)
qrels_df.head(10)

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
5,36,11705328,1
6,42,18174210,1
7,48,13734012,1
8,49,5953485,1
9,50,12580014,1


In [19]:
qrels_df['relevance'].value_counts()

relevance
1    339
Name: count, dtype: int64

# Test with querie number 1

In [20]:
queries_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   query_id    300 non-null    str  
 1   query_text  300 non-null    str  
dtypes: str(2)
memory usage: 32.2 KB


In [21]:
query = queries_df.loc[queries_df['query_id'] == '70', 'query_text'].iloc[0]

In [23]:
query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

In [24]:
scores, indices = index.search(
    query_embedding,
    k=10
)

In [25]:
for rank, idx in enumerate(indices[0], start=1):

    print(f"\nRank {rank}")
    print("Score:", scores[0][rank - 1])
    print("ID:", corpus_df.iloc[idx]["index"])


Rank 1
Score: 0.78618187
ID: 4414547

Rank 2
Score: 0.7666184
ID: 11903247

Rank 3
Score: 0.76654863
ID: 5956380

Rank 4
Score: 0.7640258
ID: 31624828

Rank 5
Score: 0.75287116
ID: 7225911

Rank 6
Score: 0.750954
ID: 21557055

Rank 7
Score: 0.747499
ID: 9483851

Rank 8
Score: 0.73814404
ID: 12240507

Rank 9
Score: 0.7373251
ID: 2582169

Rank 10
Score: 0.736439
ID: 36464673


In [26]:
qrels_df.loc[qrels_df['query_id'] == '70', 'doc_id']

15    5956380
16    4414547
Name: doc_id, dtype: str

# Evaluation Using Ranx

In [28]:
run_dict = {}

for i, row in queries_df.iterrows():
    query_id = row['query_id']  
    query_txt = row['query_text']
    
    # Get embeddings and search
    query_embedding = embedding_model.encode(
        [query_txt],
        normalize_embeddings=True
    )
    scores, indices = index.search(query_embedding, k=10)
    
    doc_scores = {}
    for rank, idx in enumerate(indices[0], start=1): 
        doc_id = corpus_df.iloc[idx]["index"]
        score = scores[0][rank - 1]
        doc_scores[doc_id] = float(score)  
    
    run_dict[query_id] = doc_scores

In [29]:
run_dict

{'1': {'4346436': 0.7425371408462524,
  '17388232': 0.7204512357711792,
  '16962732': 0.7052040696144104,
  '29638116': 0.7000970244407654,
  '927561': 0.6981079578399658,
  '21456232': 0.694959282875061,
  '23763738': 0.6948124766349792,
  '2121272': 0.691043496131897,
  '14103509': 0.6909676790237427,
  '16736872': 0.6894996762275696},
 '3': {'4414547': 0.7115472555160522,
  '10944947': 0.709050714969635,
  '19058822': 0.6994143128395081,
  '23389795': 0.696605920791626,
  '14717500': 0.6939852237701416,
  '2739854': 0.6827147006988525,
  '4632921': 0.6799712181091309,
  '10145528': 0.6767180562019348,
  '3823862': 0.6738227605819702,
  '13791788': 0.6707893013954163},
 '5': {'13734012': 0.7466669082641602,
  '27063470': 0.728550374507904,
  '25261168': 0.6832140684127808,
  '18617259': 0.6673370003700256,
  '23124332': 0.6667923927307129,
  '11349166': 0.6647814512252808,
  '21550246': 0.6647285223007202,
  '1583041': 0.6593499183654785,
  '841371': 0.6496787071228027,
  '19945096':

In [30]:
from ranx import Qrels, Run, evaluate

results = evaluate(
    qrels,
    run_dict,
    [
        "recall@5",
        "recall@10",
        "mrr@10",
        "ndcg@10"
    ]
)

In [31]:
print(results)

{'recall@5': np.float64(0.8142777777777778), 'recall@10': np.float64(0.8668888888888889), 'mrr@10': np.float64(0.7129867724867726), 'ndcg@10': np.float64(0.7469604744966187)}


In [32]:
import pandas as pd
from ranx import Qrels, Run, evaluate

# Create Qrels and Run objects
qrels_obj = Qrels(qrels)
run_obj = Run(run_dict)

# Get overall results
overall_results = evaluate(
    qrels_obj,
    run_obj,
    ["recall@5", "recall@10", "mrr@10", "ndcg@10"]
)

# Get per-query results by iterating over each query
per_query_results = {}

for query_id in qrels.keys():
    if query_id in run_dict:
        # Create single-query Qrels and Run
        single_qrels = Qrels({query_id: qrels[query_id]})
        single_run = Run({query_id: run_dict[query_id]})
        
        # Evaluate for this single query
        single_results = evaluate(
            single_qrels,
            single_run,
            ["recall@5", "recall@10", "mrr@10", "ndcg@10"]
        )
        
        per_query_results[query_id] = single_results

df_per_query = pd.DataFrame(per_query_results).T
df_per_query.index.name = 'query_id'
df_per_query = df_per_query.reset_index()

In [34]:
df_per_query

,query_id,recall@5,recall@10,mrr@10,ndcg@10
0,1,0.0,0.0,0.000000,0.000000
1,3,1.0,1.0,0.200000,0.386853
2,5,1.0,1.0,1.000000,1.000000
3,13,0.0,1.0,0.166667,0.356207
4,36,0.5,0.5,0.250000,0.264068
...,...,...,...,...,...
295,1379,1.0,1.0,1.000000,1.000000
296,1382,1.0,1.0,1.000000,1.000000
297,1385,1.0,1.0,1.000000,1.000000
298,1389,1.0,1.0,1.000000,1.000000


# Direct query expansion using LLM

In [ ]:
import os
import google.generativeai as genai
genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel('gemini-3.5-flash')

In [35]:
EXPANSION_PROMPT = """
You are an information retrieval query expansion system.

Your task is to generate a small set of expansion terms or short phrases
for a scientific search query used in dense document retrieval.

Generate ONLY additional terms or short phrases that express the SAME
concepts already present in the original query.

You may add:
- direct synonyms
- abbreviations
- alternative scientific terminology
- equivalent scientific terms or phrases

Do NOT add:
- broader concepts
- narrower or specific subtypes not explicitly mentioned
- consequences or effects
- applications
- examples
- related but distinct concepts
- new facts or information
- terms that change or reinterpret the original claim
- unnecessary or redundant terms

IMPORTANT:
- Do NOT rewrite the original query.
- Do NOT generate a new query or sentence.
- Do NOT repeat the original query.
- Do NOT explain anything.
- Generate ONLY the additional expansion terms or short phrases.
- Keep the expansion concise.
- Generate at most 5-8 terms or short phrases.
- Prefer scientifically meaningful multi-word phrases when appropriate.
- Return the terms or phrases separated by commas.

The original query will be preserved separately and automatically concatenated
with your output. Therefore, you must NOT include the original query in your output.

Original query:
{query}
"""

In [36]:
import time

def expand_query(query, max_retries=5):
    """Expand a search query using Gemini"""
    prompt = EXPANSION_PROMPT.format(query=query)
    
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            return response.text.strip()
            
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print("Maximum retries reached. Returning None.")
                return None

In [ ]:
import pandas as pd
import time

expanded_queries = []

for _, row in queries_df.iterrows():

    query_id = str(row["query_id"])
    query = row["query_text"]

    try:
        expanded_query = expand_query(query)

        expanded_queries.append({
            "query_id": query_id,
            "original_query": query,
            "expanded_query": expanded_query
        })

        print(f"Done: {query_id}")
        pd.DataFrame(expanded_queries).to_csv(
            "scifact_gemini_expanded_queries.csv",
            index=False
        )
        time.sleep(1)

    except Exception as e:
        print(f"Error for query {query_id}: {e}")

In [37]:
expanded_queries_df = pd.read_csv(
    "scifact_gemini_expanded_queries.csv",
    dtype={"query_id": str}
)

print(expanded_queries_df.shape)
expanded_queries_df

FileNotFoundError: [Errno 2] No such file or directory: 'scifact_gemini_expanded_queries.csv'